PRACTICA CALIDAD DE DATOS

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine

load_dotenv()

#conexion a bd
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

In [2]:
empleados     = pd.read_sql("SELECT * FROM empleados", engine)#probe que se puedan obtener los empleados correctamente
empleados #aqui ya tengo y muestro el dataframe

,id_empleado,nombre,puesto,fecha_contratacion,salario,direccion,correo
0,1,Juaan Perez,Gerent,2010-05-10,80000.0,"Calle Reforma #123, Col. Centro",juanperez@softwaresys.com
1,2,Maria Lopéz,Venedorr,None,45000.0,"Av. Insurgentes #456, Col. Roma",marialopez@softwaresys
2,3,Luis Gracia,NULL,2015-03-15,50000.0,"Calle Juarez #78, Col. Del Valle",NaN
3,4,Pedro Hernandéz,Vendedorr,2013-11-20,70000.0,"Blvd. Hidalgo #100, Col. Morelos",pedro@softwaresis.com
4,5,Juan Perez,Gerente,2010-05-10,60000.0,"Calle Reforma #123, Col. Centro",juanperez@softwaresys.com
5,6,Ana Martines,Gerente,2025-01-01,95000.0,"Av. Universidad #999, Col. Polanco",ana@softwaresys..com
6,7,Carlos Ramíres,Secretaría,2016-08-22,50000.0,"Calle Madero #50, Col. Centro",carlosramires@softwaresys.com


Correcion de puestos (errores de ortografia)

In [3]:
#aqui tenemos que ver primero todas las multiples opciones que se tienen registradas
empleados["puesto"].unique()

<StringArray>
['Gerent', 'Venedorr', 'NULL', 'Vendedorr', 'Gerente', 'Secretaría']
Length: 6, dtype: str

In [4]:
#entonces aqui ya entra nuestro criterio clasificando las opciones que tenemos y que valor seria el correcto
mapeo_puestos = {
    'Gerent':      'Gerente',
    'Venedorr':    'Vendedor',
    #en el caso de null no lo puedo indicar en este paso porque no es un error de ortografia, despues se indicara que puesto sera el correcto al caso o casos donde se tiene null
    'Vendedorr':   'Vendedor',
}

In [5]:
empleados['puesto'] = empleados['puesto'].replace(mapeo_puestos) #aqui es donde aplico el mapeo
print("Puestos únicos tras corrección:", empleados['puesto'].unique())

Puestos únicos tras corrección: <StringArray>
['Gerente', 'Vendedor', 'NULL', 'Secretaría']
Length: 4, dtype: str


In [6]:
empleados

,id_empleado,nombre,puesto,fecha_contratacion,salario,direccion,correo
0,1,Juaan Perez,Gerente,2010-05-10,80000.0,"Calle Reforma #123, Col. Centro",juanperez@softwaresys.com
1,2,Maria Lopéz,Vendedor,None,45000.0,"Av. Insurgentes #456, Col. Roma",marialopez@softwaresys
2,3,Luis Gracia,NULL,2015-03-15,50000.0,"Calle Juarez #78, Col. Del Valle",NaN
3,4,Pedro Hernandéz,Vendedor,2013-11-20,70000.0,"Blvd. Hidalgo #100, Col. Morelos",pedro@softwaresis.com
4,5,Juan Perez,Gerente,2010-05-10,60000.0,"Calle Reforma #123, Col. Centro",juanperez@softwaresys.com
5,6,Ana Martines,Gerente,2025-01-01,95000.0,"Av. Universidad #999, Col. Polanco",ana@softwaresys..com
6,7,Carlos Ramíres,Secretaría,2016-08-22,50000.0,"Calle Madero #50, Col. Centro",carlosramires@softwaresys.com


Limpieza de nombres y con ello eliminacion de duplicados

In [7]:
empleados["nombre"].unique()

<StringArray>
[    'Juaan Perez',     'Maria Lopéz',     'Luis Gracia', 'Pedro Hernandéz',
      'Juan Perez',    'Ana Martines',  'Carlos Ramíres']
Length: 7, dtype: str

In [8]:
# Correcciones puntuales conocidas. Basicamente la misma funcion de mapeo como se hizo en puestos
mapeo_nombres = {
    'Juaan Perez':      'Juan Perez',
    'Luis Gracia':      'Luis García',
    'Ana Martines':     'Ana Martínez',
    'Carlos Ramíres':   'Carlos Ramírez',
}

In [9]:
empleados['nombre'] = empleados['nombre'].replace(mapeo_nombres) #aplico el mapeo
empleados #muestro como quedo el dataframe

,id_empleado,nombre,puesto,fecha_contratacion,salario,direccion,correo
0,1,Juan Perez,Gerente,2010-05-10,80000.0,"Calle Reforma #123, Col. Centro",juanperez@softwaresys.com
1,2,Maria Lopéz,Vendedor,None,45000.0,"Av. Insurgentes #456, Col. Roma",marialopez@softwaresys
2,3,Luis García,NULL,2015-03-15,50000.0,"Calle Juarez #78, Col. Del Valle",NaN
3,4,Pedro Hernandéz,Vendedor,2013-11-20,70000.0,"Blvd. Hidalgo #100, Col. Morelos",pedro@softwaresis.com
4,5,Juan Perez,Gerente,2010-05-10,60000.0,"Calle Reforma #123, Col. Centro",juanperez@softwaresys.com
5,6,Ana Martínez,Gerente,2025-01-01,95000.0,"Av. Universidad #999, Col. Polanco",ana@softwaresys..com
6,7,Carlos Ramírez,Secretaría,2016-08-22,50000.0,"Calle Madero #50, Col. Centro",carlosramires@softwaresys.com


In [13]:
duplicados = empleados[empleados.duplicated(subset='nombre', keep=False)]
print("DUPLICADOS DETECTADOS ===")
print(duplicados[['id_empleado', 'nombre', 'puesto', 'salario']])


DUPLICADOS DETECTADOS ===
   id_empleado      nombre   puesto  salario
0            1  Juan Perez  Gerente  80000.0
4            5  Juan Perez  Gerente  60000.0


In [19]:
# los rangos de slario validos por puesto
rangos_salario = {
    'Gerente':    (70000, 90000),
    'Vendedor':   (30000, 50000),
    'Secretaria': (25000, 40000),
}

def salario_valido(row):
    # Si no tiene puesto o no tiene salario registrado, no podemos
    # validar nada — directo inválido
    if pd.isnull(row['puesto']) or pd.isnull(row['salario']):
        return False

    # Buscamos ese puesto en nuestro diccionario de rangos.
    # Si el puesto existe (ej. "Gerente"), rango será (70000, 90000).
    # Si el puesto no existe en el diccionario (ej. "NULL" o un valor
    # raro que no mapeamos), rango será None
    rango = rangos_salario.get(row['puesto'])

    # Si el puesto no está en nuestro diccionario, tampoco podemos
    # validar — inválido
    if not rango:
        return False

    # Si llegamos hasta aquí, tenemos puesto válido y salario.
    # Verificamos si el salario cae dentro del rango permitido
    return rango[0] <= row['salario'] <= rango[1]

empleados['salario_valido'] = empleados.apply(salario_valido, axis=1)

# Para cada grupo de duplicados, quedarse con el que tiene salario válido.
# Si ninguno lo tiene, quedarse con el de mayor salario.
def resolver_duplicado(grupo):
    validos = grupo[grupo['salario_valido']]
    if not validos.empty:
        return validos.iloc[0]
    return grupo.sort_values('salario', ascending=False).iloc[0]

empleados_sin_dup = (
    empleados
    .groupby('nombre', group_keys=False)
    .apply(resolver_duplicado, include_groups=False)
    .reset_index()
    .drop(columns='salario_valido')
)

print("EMPLEADOS SIN DUPLICADOS ===")
print(empleados_sin_dup[['id_empleado', 'nombre', 'puesto', 'salario']])

EMPLEADOS SIN DUPLICADOS ===
   id_empleado           nombre      puesto  salario
0            6     Ana Martínez     Gerente  95000.0
1            7   Carlos Ramírez  Secretaría  50000.0
2            1       Juan Perez     Gerente  80000.0
3            3      Luis García        NULL  50000.0
4            2      Maria Lopéz    Vendedor  45000.0
5            4  Pedro Hernandéz    Vendedor  70000.0


REASIGNAR VENTAS DEL ID ELIMINADO AL ID CORRECTO

In [21]:
ventas= pd.read_sql("SELECT * FROM ventas", engine)
ventas#como vemos en el dataframe se tiene referencia con el empleado 5 que ya eliminiamos

,id_venta,id_empleado,monto,fecha
0,1,1,25000.0,2022-05-10
1,2,2,-5000.0,2021-10-20
2,3,3,15000.0,2023-07-30
3,4,5,25000.0,2022-05-10
4,5,6,30000.0,2025-12-01
5,6,7,NaN,2020-03-15


In [24]:
# ID 5 (duplicado eliminado) → sus ventas pasan al ID 1

ids_validos = set(empleados_sin_dup['id_empleado'])
mapeo_ids = {}

for _, row in duplicados.iterrows():
    if row['id_empleado'] not in ids_validos:
        # Buscar el ID correcto (el que se conservo con ese nombre)
        id_correcto = empleados_sin_dup[
            empleados_sin_dup['nombre'] == row['nombre']
        ]['id_empleado'].values
        if len(id_correcto) > 0:
            mapeo_ids[row['id_empleado']] = id_correcto[0]

ventas['id_empleado'] = ventas['id_empleado'].replace(mapeo_ids)
print("Ventas tras reasignación:", ventas['id_empleado'].unique())

Ventas tras reasignación: [1 2 3 6 7]


Correcion de puestos segun salario

In [25]:
# Estrategia: opte en que es mejor basarse en salario y se ajusta el puesto, porque modificar el salario es algo mas delicado que cambiar el puesto

def corregir_puesto_por_salario(row):
    """
    Si el puesto actual no corresponde al salario se  busca que puesto sí corresponde a ese salario.
    Si ninguno corresponde, se ajusta el salario al rango del puesto actual.
    """
    if pd.isnull(row['puesto']) or pd.isnull(row['salario']):
        return row

    rango_actual = rangos_salario.get(row['puesto'])

    # Salario dentro del rango correcto → no hacer nada
    if rango_actual and rango_actual[0] <= row['salario'] <= rango_actual[1]:
        return row

    # Buscar qué puesto corresponde al salario actual
    for puesto, (minimo, maximo) in rangos_salario.items():
        if minimo <= row['salario'] <= maximo:
            row['puesto'] = puesto
            return row

    # Si el salario no entra en ningún rango, ajustarlo al rango del puesto
    if rango_actual:
        if row['salario'] > rango_actual[1]:
            row['salario'] = rango_actual[1]  # bajar al máximo
        elif row['salario'] < rango_actual[0]:
            row['salario'] = rango_actual[0]  # subir al mínimo

    return row

empleados_sin_dup = empleados_sin_dup.apply(corregir_puesto_por_salario, axis=1)
print("TRAS CORRECCIÓN SALARIO–PUESTO ===")
print(empleados_sin_dup[['id_empleado', 'nombre', 'puesto', 'salario']])

TRAS CORRECCIÓN SALARIO–PUESTO ===
   id_empleado           nombre    puesto  salario
0            6     Ana Martínez   Gerente  90000.0
1            7   Carlos Ramírez  Vendedor  50000.0
2            1       Juan Perez   Gerente  80000.0
3            3      Luis García  Vendedor  50000.0
4            2      Maria Lopéz  Vendedor  45000.0
5            4  Pedro Hernandéz   Gerente  70000.0


In [27]:
empleados_sin_dup = empleados_sin_dup.sort_values('id_empleado').reset_index(drop=True)
empleados_sin_dup

,nombre,id_empleado,puesto,fecha_contratacion,salario,direccion,correo
0,Juan Perez,1,Gerente,2010-05-10,80000.0,"Calle Reforma #123, Col. Centro",juanperez@softwaresys.com
1,Maria Lopéz,2,Vendedor,None,45000.0,"Av. Insurgentes #456, Col. Roma",marialopez@softwaresys
2,Luis García,3,Vendedor,2015-03-15,50000.0,"Calle Juarez #78, Col. Del Valle",NaN
3,Pedro Hernandéz,4,Gerente,2013-11-20,70000.0,"Blvd. Hidalgo #100, Col. Morelos",pedro@softwaresis.com
4,Ana Martínez,6,Gerente,2025-01-01,90000.0,"Av. Universidad #999, Col. Polanco",ana@softwaresys..com
5,Carlos Ramírez,7,Vendedor,2016-08-22,50000.0,"Calle Madero #50, Col. Centro",carlosramires@softwaresys.com
